# Phase 1 — Notebook p1_01: Tokenizer Training

**What this notebook does:** trains the two tokenizers the whole capstone
runs on, compares them, and publishes both as a Kaggle Dataset.

- **T1 — byte-level BPE**, trained with our own merge logic (same family as
  the CS781 Assignment-1 BPE, retrained on legal text). Used by models A, B
  and S. Byte-level means it can encode any input with zero `<unk>` tokens,
  and digit/punctuation splitting is tuned so entity boundaries land on
  clean token boundaries — this matters a lot in Phase 2 (NER scores on
  exact character offsets).
- **T2 — SentencePiece Unigram**, a different algorithm entirely (a
  probabilistic subword language model, pruned down to size via EM rather
  than built up via greedy merges). Used by models C and D. This isn't
  "the same tokenizer with a different vocab size" — it's a genuinely
  different segmentation strategy, which is the point of comparing them.

Both tokenizers share **one reserved special-token layout**, fixed now so
none of Phases 2-7 ever needs a vocab resize later.

**Runs on:** Kaggle CPU session. Tokenizer training is CPU work start to
finish — this notebook never touches the GPU quota.

**Needs as input:** the `legallm-phase1-data` Kaggle Dataset from
`p1_00_prep.ipynb` (Add Data -> search for it before running this).


## 1. Imports

`tokenizers` (Hugging Face's Rust-backed library) trains T1; `sentencepiece`
trains T2. Both are usually preinstalled on Kaggle images, but we check
rather than assume.


In [ ]:
import subprocess, sys

def ensure(pkg, import_name=None):
    """Import a package, installing it first if it isn't already present.
    Small helper so we don't repeat the same try/except three times."""
    try:
        return __import__(import_name or pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
        return __import__(import_name or pkg)

tokenizers_pkg = ensure("tokenizers")
sentencepiece_pkg = ensure("sentencepiece")

from tokenizers import Tokenizer, models, pre_tokenizers, trainers, decoders
import sentencepiece as spm

import json
import os
import random
import time
from collections import Counter
from pathlib import Path

print("Imports OK.")
print("tokenizers version:", tokenizers_pkg.__version__)


## 2. Config

The special-token layout here is the one **every later phase notebook
reuses verbatim** — it's the interface contract from the plan doc, not a
per-notebook detail. If this list ever needs to change, it needs to change
everywhere at once, so treat this cell as the single source of truth for it.


In [ ]:
# --- Where the previous notebook's output lives as a Kaggle input --------
INPUT_DIR = Path("/kaggle/input/legallm-phase1-data")
SOURCE_NAMES = ["acts", "high_courts", "supreme_court", "tribunals"]

# --- Where this notebook writes its own outputs ---------------------------
OUTPUT_DIR = Path("/kaggle/working/phase1_tokenizers")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Tokenizer training budget ---------------------------------------------
# Tokenizer vocabularies converge well before you'd think — a few hundred MB
# of text is plenty for a stable 32k vocab. We cap it here mostly so
# SentencePiece's EM training (the slower of the two) stays in the
# "minutes" range instead of drifting toward an hour.
CHARS_PER_SOURCE = 100_000_000  # 100MB x 4 sources = ~400MB combined sample
VOCAB_SIZE = 32_000
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

# --- The shared special-token layout for every model in every phase -------
CORE_SPECIALS = ["<pad>", "<bos>", "<eos>", "<unk>", "<mask>", "<sep>", "<cls>"]
EXTRA_ID_SENTINELS = [f"<extra_id_{i}>" for i in range(100)]   # span-corruption sentinels (T5-style, used by D)
TASK_TOKENS = ["<ner>", "<rr>", "<cjpe>", "<lsi>", "<summ>", "<pcr>", "<qa>"]  # one per capstone phase
RESERVED_TOKENS = [f"<reserved_{i}>" for i in range(20)]        # headroom for anything we forgot

ALL_SPECIAL_TOKENS = CORE_SPECIALS + EXTRA_ID_SENTINELS + TASK_TOKENS + RESERVED_TOKENS
print(f"Reserved {len(ALL_SPECIAL_TOKENS)} special tokens "
      f"(out of a {VOCAB_SIZE:,}-token vocabulary) — plenty of headroom.")


## 3. Build one shared, balanced training sample

Both tokenizers train on **exactly the same text**, sampled evenly across
the four legal sources (not proportional to file size — Acts is a small
fraction of the raw corpus, but its vocabulary is just as important to
cover as the much bigger High Courts file). One combined file, one line per
document, is simplest for both trainers to consume.


In [ ]:
def stream_docs(path):
    """Yield one cleaned document at a time from a clean_<source>.txt file.
    Documents in that file are separated by a blank line (see
    p1_00_prep.ipynb, section 8) — we just re-join lines until we hit one.
    """
    buf = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            if line.strip() == "":
                if buf:
                    yield " ".join(buf)
                    buf = []
            else:
                buf.append(line.strip())
    if buf:
        yield " ".join(buf)


combined_path = OUTPUT_DIR / "combined_sample.txt"
per_source_chars = {}

with open(combined_path, "w", encoding="utf-8") as out_f:
    for name in SOURCE_NAMES:
        src_path = INPUT_DIR / f"clean_{name}.txt"
        chars_written = 0
        t0 = time.time()
        for doc in stream_docs(src_path):
            if chars_written >= CHARS_PER_SOURCE:
                break
            out_f.write(doc)
            out_f.write("\n")
            chars_written += len(doc) + 1
        per_source_chars[name] = chars_written
        print(f"  {name:14s}: {chars_written:,} chars ({time.time() - t0:.1f}s)")

total_chars = sum(per_source_chars.values())
print(f"\nCombined training sample: {total_chars:,} chars -> {combined_path}")


## 4. Train T1 — byte-level BPE

Three deliberate choices here, each tied to something downstream:

- **Byte-level base alphabet** (256 starting tokens covering every possible
  byte) means this tokenizer never has to emit `<unk>` — any input,
  however unusual, is representable.
- **Digits split one at a time** (`1234` -> `1`, `2`, `3`, `4`) so dates and
  case numbers don't get swallowed into a handful of opaque merged tokens
  that would be awkward to align back to character offsets in Phase 2.
- **Punctuation isolated** so `.`, `,`, `(`, `)`, `/`, `§`, `-` are always
  their own tokens, never fused onto a word — again, cleaner offsets.


In [ ]:
bpe_tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))

# Pre-tokenization runs *before* BPE merging: split on digits, then isolate
# punctuation, then apply byte-level encoding (this is also what turns a
# leading space into the "Ġ"-style marker GPT-style tokenizers use).
bpe_tokenizer.pre_tokenizer = pre_tokenizers.Sequence([
    pre_tokenizers.Digits(individual_digits=True),
    pre_tokenizers.Punctuation(behavior="isolated"),
    pre_tokenizers.ByteLevel(add_prefix_space=False, use_regex=True),
])
bpe_tokenizer.decoder = decoders.ByteLevel()

bpe_trainer = trainers.BpeTrainer(
    vocab_size=VOCAB_SIZE,
    min_frequency=2,
    special_tokens=ALL_SPECIAL_TOKENS,
    show_progress=True,
)

print("Training T1 (byte-level BPE)...")
t0 = time.time()
bpe_tokenizer.train([str(combined_path)], bpe_trainer)
print(f"Done in {time.time() - t0:.1f}s. Final vocab size: {bpe_tokenizer.get_vocab_size():,}")

t1_path = OUTPUT_DIR / "t1_bpe.json"
bpe_tokenizer.save(str(t1_path))
print(f"Saved T1 -> {t1_path}")


## 5. Train T2 — SentencePiece Unigram

A genuinely different algorithm from T1, not a re-run with different
settings: Unigram starts from a large pool of candidate subwords and prunes
it down via an EM-style loss, rather than building the vocabulary up one
merge at a time. `character_coverage=0.9995` (not `1.0`) deliberately
leaves a small margin for rare Unicode characters in scanned/OCR'd
documents to fall back to byte fragments instead of forcing them all into
the vocabulary.


In [ ]:
# SentencePiece handles the four "core" control tokens through dedicated
# ids/pieces; everything else in our special-token layout goes in as plain
# user-defined symbols (never split, never merged with neighbours).
other_symbols = ["<mask>", "<sep>", "<cls>"] + EXTRA_ID_SENTINELS + TASK_TOKENS + RESERVED_TOKENS

t2_prefix = OUTPUT_DIR / "t2_unigram"

print("Training T2 (SentencePiece Unigram)...")
t0 = time.time()
spm.SentencePieceTrainer.train(
    input=str(combined_path),
    model_prefix=str(t2_prefix),
    vocab_size=VOCAB_SIZE,
    model_type="unigram",
    character_coverage=0.9995,
    input_sentence_size=2_000_000,   # cap how many lines the EM step actually sees
    shuffle_input_sentence=True,      # so the cap doesn't just mean "first N lines"
    pad_id=0, pad_piece="<pad>",
    unk_id=1, unk_piece="<unk>",
    bos_id=2, bos_piece="<bos>",
    eos_id=3, eos_piece="<eos>",
    user_defined_symbols=",".join(other_symbols),
    num_threads=os.cpu_count(),
)
print(f"Done in {time.time() - t0:.1f}s.")
print(f"Saved T2 -> {t2_prefix}.model / {t2_prefix}.vocab")


## 6. Load both tokenizers (+ a GPT-2 baseline) for comparison

We reload from disk rather than reusing the in-memory objects above — this
doubles as a smoke test that the saved files actually work on their own,
which is exactly how every later phase notebook will load them.


In [ ]:
from tokenizers import Tokenizer as HFTokenizer

t1 = HFTokenizer.from_file(str(t1_path))

t2 = spm.SentencePieceProcessor()
t2.load(str(t2_prefix) + ".model")

# The GPT-2 tokenizer is our "did we actually do better than the starter
# code's default" baseline — not something any model trains on.
gpt2_tokenizer = ensure("transformers").AutoTokenizer.from_pretrained("gpt2")

print("T1 vocab size:", t1.get_vocab_size())
print("T2 vocab size:", t2.get_piece_size())
print("GPT-2 vocab size:", len(gpt2_tokenizer))


## 7. Fertility comparison on the held-out dev set

"Fertility" = tokens produced per word. Lower is better: it means more
actual context fits inside a model's fixed context window, and it's the
right metric to compare *across* tokenizers with different vocabularies —
unlike raw per-token loss, which isn't comparable once the vocabulary
changes. We reuse the dev set from `p1_00_prep.ipynb` so this measures
fertility on real held-out legal text, not the training sample.


In [ ]:
dev_prompts_path = Path("/kaggle/input/legallm-phase1-data/dev_prompts.json")
with open(dev_prompts_path, encoding="utf-8") as f:
    dev_prompts = json.load(f)

sample_texts = [rec["text"] for rec in dev_prompts.values()]
total_words = sum(len(t.split()) for t in sample_texts)


def fertility(name, encode_fn):
    """tokens-per-word over the whole dev set, plus how many words that
    implies fit inside a fixed-size context window — the number that
    actually matters when deciding context length for models A-D.
    """
    total_tokens = sum(encode_fn(t) for t in sample_texts)
    words_per_token = total_words / total_tokens
    print(f"{name:10s}  tokens/word: {total_tokens / total_words:5.3f}   "
          f"words fitting in a 2048-token window: {2048 * words_per_token:7,.0f}")


print(f"Dev set: {len(sample_texts)} documents, {total_words:,} words total\n")
fertility("T1 (BPE)", lambda t: len(t1.encode(t).ids))
fertility("T2 (Unigram)", lambda t: len(t2.encode(t, out_type=int)))
fertility("GPT-2", lambda t: len(gpt2_tokenizer.encode(t)))


## 8. Sanity checks

A handful of checks that would each have caught a real bug while this
notebook was being written: every special token actually resolves to its
own id (not silently split into pieces), round-trip decoding doesn't mangle
the text, and neither tokenizer is quietly falling back to `<unk>` a lot on
ordinary legal text.


In [ ]:
print("=== Special-token round trip (T1) ===")
for tok in ["<pad>", "<extra_id_0>", "<ner>", "<reserved_0>"]:
    tid = t1.token_to_id(tok)
    assert tid is not None, f"{tok} did not get its own id in T1!"
print(f"All {len(ALL_SPECIAL_TOKENS)} special tokens present in T1.  (e.g. '<ner>' -> id {t1.token_to_id('<ner>')})")

print("\n=== Special-token round trip (T2) ===")
for tok in ["<pad>", "<extra_id_0>", "<ner>", "<reserved_0>"]:
    tid = t2.piece_to_id(tok)
    assert tid != t2.unk_id(), f"{tok} fell back to <unk> in T2!"
print(f"All spot-checked special tokens present in T2.  (e.g. '<ner>' -> id {t2.piece_to_id('<ner>')})")

print("\n=== Decode round trip on a real dev sentence ===")
sample = sample_texts[0][:300]
t1_decoded = t1.decode(t1.encode(sample).ids)
t2_decoded = t2.decode(t2.encode(sample, out_type=int))
print("Original :", sample[:120].replace(chr(10), " "))
print("T1 decode:", t1_decoded[:120].replace(chr(10), " "))
print("T2 decode:", t2_decoded[:120].replace(chr(10), " "))

print("\n=== <unk> rate on the dev set (should be near zero for both) ===")
t1_unk_id = t1.token_to_id("<unk>")
t1_unk_count = sum(t1.encode(t).ids.count(t1_unk_id) for t in sample_texts[:200])
t2_unk_count = sum(t2.encode(t, out_type=int).count(t2.unk_id()) for t in sample_texts[:200])
print(f"T1 <unk> count over 200 dev docs: {t1_unk_count}")
print(f"T2 <unk> count over 200 dev docs: {t2_unk_count}")

print("\nAll checks passed.")


## 9. Publish as a Kaggle Dataset

Same pattern as the data-prep notebook's push cell — reused as-is now that
it's confirmed working, just pointed at a new dataset slug.


In [ ]:
KAGGLE_USERNAME = "jatiniitk"
DATASET_SLUG = "legallm-tokenizers"

from kaggle_secrets import UserSecretsClient

try:
    kaggle_token = UserSecretsClient().get_secret("KAGGLE_API_TOKEN")
except Exception as exc:
    kaggle_token = None
    print(f"Could not read the 'KAGGLE_API_TOKEN' secret ({exc}). "
          f"Add it under Add-ons -> Secrets before running this cell.")

if kaggle_token:
    kaggle_dir = Path.home() / ".kaggle"
    kaggle_dir.mkdir(exist_ok=True)
    token_path = kaggle_dir / "access_token"
    token_path.write_text(kaggle_token)
    token_path.chmod(0o600)

    dataset_id = f"{KAGGLE_USERNAME}/{DATASET_SLUG}"
    metadata = {"title": DATASET_SLUG, "id": dataset_id, "licenses": [{"name": "CC0-1.0"}]}
    with open(OUTPUT_DIR / "dataset-metadata.json", "w") as f:
        json.dump(metadata, f)

    # We don't need combined_sample.txt in the published dataset — it was
    # only scratch input for training, not something any later notebook
    # loads. Deleting it keeps the published dataset small and fast to
    # attach elsewhere.
    combined_path.unlink(missing_ok=True)

    exists = subprocess.run(
        ["kaggle", "datasets", "status", dataset_id], capture_output=True, text=True,
    ).returncode == 0

    if exists:
        print(f"Dataset {dataset_id} already exists -> pushing a new version.")
        cmd = ["kaggle", "datasets", "version", "-p", str(OUTPUT_DIR),
               "-m", f"Tokenizer training run at {time.strftime('%Y-%m-%d %H:%M:%S')}", "-r", "zip"]
    else:
        print(f"Dataset {dataset_id} does not exist yet -> creating it.")
        cmd = ["kaggle", "datasets", "create", "-p", str(OUTPUT_DIR), "-r", "zip"]

    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr)
    else:
        print(f"Done. Attach it to future notebooks via 'Add Data' -> search '{DATASET_SLUG}'.")
else:
    print("Skipped: missing KAGGLE_API_TOKEN secret.")


## 10. Summary

| File | What it is |
| --- | --- |
| `t1_bpe.json` | T1 — load with `tokenizers.Tokenizer.from_file(...)`, used by models A, B, S |
| `t2_unigram.model` / `.vocab` | T2 — load with `sentencepiece.SentencePieceProcessor().load(...)`, used by models C, D |

**Next notebook:** `p1_02_bench.ipynb` — a short kernel/throughput benchmark
(SDPA backends on T4, whether FlexAttention compiles, batch-size sweep)
before committing 6-12 GPU-hours to the real architecture runs.

**Reminder for every later phase notebook:** load a tokenizer by attaching
`legallm-tokenizers` as an input and reading straight from these two files
— never retrain a tokenizer inside a phase notebook, and never let a
phase's fine-tuning head add new tokens outside the reserved layout in
Section 2 above.
